# Evaluation

#### Why this phase exists

Most beginners build a RAG app or an agent, see it answer one question correctly, and declare victory.

Then it fails silently in production and nobody notices for weeks.

```
The real question is never "did it answer?"
It's "is it RIGHT, often enough, for the cases we care about?"
```

#### The mental model: unit tests for AI

As a backend developer you already trust this idea:

```
Code change  →  run tests  →  green = safe to ship
```

Evaluation is the same discipline for AI:

```
Prompt / model / pipeline change
        ↓
   run evals
        ↓
   scores go up or down?
        ↓
   ship or roll back
```

The twist: AI outputs aren't simply "equal / not equal," so we need softer measures. This notebook covers the core ones.

#### Goal

***Learn how companies decide whether an AI app is actually working.***

### What You Measure Depends on the Task

Before picking a metric, ask what **kind** of task you're evaluating. The right metric is different for each.

```
Classification / extraction  →  exact match, accuracy
Retrieval (RAG)              →  precision / recall
Generation (free text)       →  relevance, faithfulness, LLM-judge
```

```
Wrong metric for the task = misleading scores.
```

For example, "exact match" is perfect for *"which category is this ticket?"* but useless for *"summarize this complaint"* — two valid summaries can be worded completely differently.

The rest of this notebook walks these from the **simplest** (exact match) to the **fuzziest** (LLM-as-a-judge).

### Ground Truth

Ground truth is the **known correct answer** you measure against — the "expected" in an assertion.

```
assertEqual(actual, expected)
                     ▲
                ground truth
```

You build an **evaluation dataset**: a list of inputs paired with their correct outputs.

```
[
  {
    "question": "How many annual leave days do employees get?",
    "expected": "20 days",
    "expected_source": "handbook.pdf, page 12"
  },
  {
    "question": "What is the WFH policy?",
    "expected": "Up to 3 days per week with manager approval"
  }
]
```

#### Where ground truth comes from

```
- Hand-written by experts / the team
- Real historical tickets with known answers
- Curated from documentation
```

```
No ground truth  →  no objective way to say "this is better."
```

It's tedious to build, which is exactly why beginners skip it — and why their systems quietly degrade.

### The Simplest Eval: Exact / Deterministic Checks

When the correct answer is fixed, evaluation is just an **assertion** — the same thing you already do in unit tests.

```
Exact match   →  output == expected
Contains      →  expected in output
Regex / rule  →  output matches a pattern
Valid schema  →  output parses into the required shape (Phase 6)
```

Examples:

```
Task: classify sentiment
  expected: "positive"
  output:   "positive"      →  PASS (exact match)

Task: extract an order id
  expected: "ORD-12345"
  output:   "ORD-12345"     →  PASS

Task: structured output
  check:    json.loads(output) has keys name, email, age  →  PASS
```

When this works, **use it** — it's free, fast, and 100% objective.

```
Deterministic task?  →  assertion-style eval.
Open-ended task?     →  you need the softer metrics below.
```

### Precision & Recall

These measure **retrieval quality** in RAG (Phase 4): when you fetch the top-K chunks, were they the right ones?

```
Precision = of the chunks we RETURNED, how many were relevant?
Recall    = of all the relevant chunks that EXIST, how many did we find?
```

#### Example

There are **4** truly relevant chunks in the knowledge base. Your system returns **5** chunks, of which **3** are relevant.

```
Returned:        5 chunks
Relevant found:  3
Relevant total:  4
```

```
Precision = 3 / 5 = 0.60   (60% of what we showed was useful)
Recall    = 3 / 4 = 0.75   (we found 75% of what mattered)
```

#### The trade-off

```
High precision, low recall  →  "only shows great results, but misses some"
High recall, low precision  →  "finds everything, but lots of noise"
```

For RAG, **recall** is often critical: if the relevant chunk is never retrieved, the LLM literally cannot answer correctly — no matter how good the prompt is.

```
Bad retrieval  →  bad answer, guaranteed.
Evaluate retrieval BEFORE blaming the LLM.
```

### F1 Score

Precision and recall pull in opposite directions, so it helps to have **one number** that balances them.

```
F1 = harmonic mean of precision and recall
```

Using the previous example:

```
Precision = 0.60
Recall    = 0.75

F1 = 2 * (0.60 * 0.75) / (0.60 + 0.75)
   = 2 * 0.45 / 1.35
   ≈ 0.67
```

#### Why harmonic mean (not a plain average)?

Because it punishes lopsided scores:

```
Precision 1.0, Recall 0.0
  plain average = 0.50   (looks okay — misleading)
  F1            = 0.00   (correctly says: useless)
```

Use **F1** when you want a single quality number; look at **precision and recall separately** when you care about the trade-off.

### Relevance

Relevance asks a simpler question:

```
Does the answer (or retrieved context) actually address what was asked?
```

Example of a **relevant but unhelpful** drift:

```
Question: "How do I reset my password?"

Answer:   "Passwords are important for security and should be
           changed regularly using strong characters."
```

True, on-topic-ish... but it never answers the question. **Low relevance.**

```
Relevant answer:
"Go to Settings → Security → Reset Password, then check your email."
```

Relevance is often scored on a scale (e.g. 1–5) or yes/no, frequently by an LLM judge (coming up). It catches answers that *sound* related but don't satisfy the user's intent.

### Faithfulness (Groundedness)

This is the **anti-hallucination** metric, and it's central to RAG.

```
Faithfulness = Is the answer actually SUPPORTED by the retrieved context?
```

A faithful answer only states things the context backs up. An unfaithful answer adds invented "facts."

#### Example

```
Context: "Employees receive 20 annual leave days."

Question: "How many leave days and can I carry them over?"

Faithful answer:
  "You receive 20 annual leave days. The context doesn't
   mention carry-over."

Unfaithful answer:
  "You receive 20 annual leave days and can carry over 10."
                                          ▲
                          invented — not in the context
```

#### Relevance vs Faithfulness (don't confuse them)

```
Relevance     →  is the answer ABOUT the question?
Faithfulness  →  is the answer SUPPORTED by the source?
```

An answer can be relevant but unfaithful (on-topic hallucination) — the most dangerous kind, because it sounds right.

### LLM-as-a-Judge

Problem: metrics like relevance and faithfulness need *human judgement* — but checking thousands of outputs by hand doesn't scale.

Solution: use a **second LLM to grade the first one's output**.

```
Question + Answer (+ Context)
        ↓
   Judge LLM  ── given a rubric ──▶  score + reason
```

#### Example judge prompt

```
You are an evaluator.

Question:
{question}

Context:
{retrieved_context}

Answer:
{answer}

Score FAITHFULNESS from 1-5:
5 = every claim is supported by the context
1 = the answer invents facts not in the context

Respond as JSON:
{ "score": <1-5>, "reason": "<short explanation>" }
```

(Notice this reuses Phase 5 + Phase 6: a structured prompt with a structured output.)

#### Strengths and cautions

```
✓ Scales to thousands of cases
✓ Handles fuzzy criteria humans use
✓ Cheaper and faster than human review

✗ The judge can be biased or wrong
✗ Validate the judge against some human labels
✗ Don't judge with the SAME model that's being tested, when avoidable
```

LLM-as-a-judge is the workhorse of modern AI evaluation — but treat the judge as *another component that itself needs checking.*

### Caveat: LLM Outputs Aren't Deterministic

A surprise that confuses beginners: run the **same input twice** and the answer can differ. That's the sampling from Phase 1 (temperature / top-p) at work.

```
Same prompt
  ↓
run 1 → "You get 20 leave days."
run 2 → "Employees are entitled to 20 days of annual leave."
```

Both correct — different text. This breaks naive exact-match eval and makes scores wobble between runs.

#### How to handle it during evaluation

```
* Lower temperature (→ 0) for repeatable, gradeable outputs
* OR run each case several times and average the score
* Use meaning-based metrics (relevance / faithfulness / judge),
  not exact string match, for open-ended answers
```

```
Don't panic when a score moves a little between runs —
some variance is expected. Watch TRENDS, not single runs.
```

### Putting It Together: An Eval Pipeline

```
Eval dataset (questions + ground truth)
        ↓
Run each question through YOUR system
        ↓
For each output, compute:
   - retrieval:  precision / recall
   - answer:     relevance, faithfulness, correctness
        ↓
Aggregate into scores
        ↓
Compare against the previous version
        ↓
   better? → ship      worse? → investigate / roll back
```

#### Offline vs online evaluation

```
Offline  →  run on a fixed dataset before shipping (like CI tests)
Online   →  measure REAL traffic after shipping
            (user thumbs up/down, complaint rate, etc.)
```

#### The golden rule

```
Change ONE thing at a time
(prompt OR model OR chunk size OR k)
        ↓
re-run evals
        ↓
keep the change only if scores improve
```

Without this, "improving" an AI app is just vibes. With it, it's engineering.

### Beyond Quality: Latency & Cost

Correctness isn't the only thing production cares about. Two answers can be equally correct but very different to ship.

```
Quality   →  is it right?        (everything above)
Latency   →  how fast?           (seconds per response)
Cost      →  how many tokens?    (Phase 1 — tokens = money)
```

A real evaluation usually tracks all three together:

```
Model A:  92% faithful,  1.2s,  $0.004 / request
Model B:  94% faithful,  6.5s,  $0.03  / request
```

Is B's +2% quality worth ~5x slower and ~7x pricier? That's a product decision — but you can only make it if you **measure** all three.

```
Best AI app  !=  highest quality score.
Best  =  the right balance of quality, speed, and cost.
```

### Summary & Goal

```
Ground Truth   →  the known-correct answer to compare against
Exact Match    →  literal/assertion check for deterministic outputs
Precision      →  of what we returned, how much was relevant
Recall         →  of all relevant items, how many we found
F1             →  one score balancing precision and recall
Relevance      →  does the answer address the question?
Faithfulness   →  is the answer supported by the source (no hallucination)?
LLM-as-a-Judge →  use an LLM to grade outputs at scale
```

Two reminders: pick the metric that matches the **task**, and remember quality isn't the only axis — also track **latency** and **cost**.

#### Goal

***Know whether your AI app actually works — with evidence, not vibes.***

Think of this as **unit tests for AI systems**. You'll lean on it heavily once your apps start *acting* on their own — which is exactly what happens next in **Phase 9 — Agents**.